# 02 — Source download and resume

Owner: Member 2. Runs on Google Colab CPU. Takes the paper IDs and version counts found by
`01_discovery.ipynb`, downloads every source version from `export.arxiv.org/src/{id}vN`, keeps
**only `.tex` and `.bib` files**, and saves progress on Google Drive so a disconnect loses nothing.

Functions provided to the rest of the pipeline:

| Function | Returns |
|---|---|
| `assigned_ids()` | `list[str]`: the 20,000 IDs in `ID_RANGES` |
| `load_manifest(path)` / `validate_assigned_manifest(records)` | normalized discovery records (`arxiv_id`, `version_count`, ...) |
| `extract_source(archive, folder)` | writes the `.tex`/`.bib` files of one archive; returns sizes and checksums |
| `download_sources(manifest, output, state, ...)` | downloads / resumes every version; returns the run summary |

Output: `DATA_DIR/<yymm-id>/vN/` (only `.tex`/`.bib`). State: `STATE_DIR/download/`.

This notebook only defines the downloader and its offline tests, like `01_discovery.ipynb`.
The full crawl is started from `05_run_pipeline.ipynb`.

### How it works

1. **One paced connection.** `SourceClient` sends one request at a time, at least `ARXIV_DELAY`
   (3 s) apart, retries 429/5xx with backoff and honours `Retry-After`.
2. **Pipeline.** The main thread only downloads, to the fast local disk. Extraction and the slow
   writes to Drive run in `DOWNLOAD_WORKERS` background threads, so they overlap the 3 s wait
   instead of adding to it. At most `2 × DOWNLOAD_WORKERS` fetched archives wait for processing.
3. **Only `.tex` and `.bib`.** Archives are unpacked on local disk and only `.tex`/`.bib` are copied
   to Drive. Figures, `.bbl`, `.sty`, `.cls` and the raw archive never reach Drive. A version with no
   `.tex` (PDF-only) gets an empty `vN/` folder and an `empty_source` receipt.
4. **Fast resume.** Every finished version gets a receipt (`receipts/<id>/vN.json`, also appended
   to `receipts.jsonl`). At start-up the index is read once and saved files are checked by size
   (`DOWNLOAD_VERIFY = "sha256"` re-hashes them instead). Files other than `.tex`/`.bib` kept by
   older runs are deleted during this check, without new requests.

**Error handling.** A failed version is written to `errors.jsonl` and retried on the next run.
Unsafe archive paths (`../`, links, a file used as a folder) and HTML/JSON error pages are rejected.
Files whose names differ only in case are both kept (renamed `name~2.tex` on a case-insensitive disk).
On Ctrl+C or a Colab stop, archives that were already downloaded are processed before exiting.

In [ ]:
if "CONFIG_LOADED" not in globals():
    get_ipython().run_line_magic("run", "-i ./00_config.ipynb")

import bz2
import errno
from concurrent.futures import FIRST_COMPLETED, ThreadPoolExecutor, wait
from email.utils import parsedate_to_datetime
import gzip
import hashlib
import lzma
from pathlib import PurePosixPath
import shutil
import socket
import tarfile
import tempfile
import threading
import uuid
import zipfile

import requests

try:
    from tqdm.auto import tqdm   # progress bars (widgets in Jupyter / Colab, text elsewhere)
except ImportError:
    tqdm = None                  # no bars: plain progress lines instead

try:
    import psutil
except ImportError:  # The downloader still works without RAM sampling.
    psutil = None

## 1. Settings

`ARXIV_CONTACT` (optional, an email address) is added to the User-Agent. `DOWNLOAD_WORKERS` and
`DOWNLOAD_VERIFY` are explained in the introduction.

In [ ]:
KEEP = frozenset({".tex", ".bib"})      # the only files written to Drive
ID_PATTERN = re.compile(r"^\d{2}(?:0[1-9]|1[0-2])\.\d{4,5}$")
RETRY_STATUSES = frozenset({429, 500, 502, 503, 504})
MAX_TEXT_BYTES = 128 * 1024 * 1024       # safety limit for one .tex/.bib file
TEX_COMMAND = re.compile(rb"\\(?:documentclass|documentstyle|begin|input|def|newcommand|section)\b")

DOWNLOAD_STATE = STATE_DIR / "download"   # receipts, errors, progress, papers.csv
DOWNLOAD_WORKERS = 2        # threads that extract archives and write to Drive
DOWNLOAD_VERIFY = "size"    # resume check: "size" (fast) or "sha256" (re-hash every saved file)

_contact = os.environ.get("ARXIV_CONTACT", "").strip()
if _contact and "@" not in _contact:
    raise ValueError("ARXIV_CONTACT must be an email address when set.")
DOWNLOAD_USER_AGENT = USER_AGENT + (f" (mailto:{_contact})" if _contact else "")
DOWNLOAD_STATE.mkdir(parents=True, exist_ok=True)

## 2. Discovery manifest

A JSON list (or `{"papers": [...]}`) with `id`/`versions` from `01_discovery`; `arxiv_id`/`version_count`
is also accepted. Duplicate IDs and impossible version counts are rejected.

In [ ]:
def _now():
    return datetime.now(timezone.utc).isoformat()


def _atomic_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + "." + uuid.uuid4().hex + ".tmp")
    try:
        with temporary.open("w", encoding="utf-8") as stream:
            json.dump(value, stream, ensure_ascii=False, indent=2)
            stream.write("\n")
            stream.flush()
            os.fsync(stream.fileno())
        os.replace(temporary, path)
    finally:
        temporary.unlink(missing_ok=True)


def validate_manifest(records):
    """Normalize discovery records, rejecting ambiguous IDs and counts."""
    if not isinstance(records, list) or not records:
        raise ValueError("Manifest must contain a nonempty list of papers")
    normalized, seen = [], set()
    for item in records:
        if not isinstance(item, dict):
            raise ValueError("Each paper must be an object")
        identifier = item.get("arxiv_id", item.get("id"))
        count = item.get("version_count", item.get("versions"))
        if not isinstance(identifier, str) or not ID_PATTERN.fullmatch(identifier) or int(identifier.split(".")[1]) == 0:
            raise ValueError(f"Invalid modern arXiv ID: {identifier!r}")
        if identifier in seen:
            raise ValueError(f"Duplicate arXiv ID: {identifier}")
        if type(count) is not int or count < 0 or (count == 0 and item.get("status") not in ("not_found", "error")):
            raise ValueError(f"Invalid version count for {identifier}: {count!r}")
        seen.add(identifier)
        normalized.append(dict(item, arxiv_id=identifier, version_count=count))
    return normalized


def load_manifest(path):
    with Path(path).open(encoding="utf-8-sig") as stream:
        document = json.load(stream)
    return validate_manifest(document.get("papers") if isinstance(document, dict) else document)


def assigned_ids():
    """The exact 20,000 IDs in ID_RANGES (00_config), in instructor order."""
    ids = []
    for start, end in ID_RANGES:
        month, low = start.split(".")
        ids += [f"{month}.{n:0{len(low)}d}" for n in range(int(low), int(end.split(".")[1]) + 1)]
    return ids


def validate_assigned_manifest(records):
    normalized = validate_manifest(records)
    if {record["arxiv_id"] for record in normalized} != set(assigned_ids()):
        raise ValueError("Discovery manifest must cover exactly the 20,000 assigned IDs")
    return normalized

## 3. Paced source client

All attempts, retries included, share one rate gate. A 429/5xx response pushes the next request
back by `max(2^attempt, Retry-After)` seconds; 404 and other errors are not retried.

In [ ]:
class SourceClient:
    """Single-threaded arXiv source client; retry attempts share the rate gate."""

    def __init__(self, session=None, user_agent=USER_AGENT, delay=ARXIV_DELAY,
                 max_attempts=4, clock=time.monotonic, sleeper=time.sleep):
        if delay < 3:
            raise ValueError("arXiv delay must be at least 3 seconds")
        if max_attempts < 1:
            raise ValueError("max_attempts must be positive")
        self.session = session if session is not None else requests.Session()
        self.session.headers.update({"User-Agent": user_agent, "Accept-Encoding": "identity"})
        self.delay, self.max_attempts = delay, max_attempts
        self.clock, self.sleeper = clock, sleeper
        self.last_start = None
        self.not_before = 0.0

    def _pace(self):
        wait = max(0.0, self.not_before - self.clock())
        if self.last_start is not None:
            wait = max(wait, self.delay - (self.clock() - self.last_start))
        if wait:
            self.sleeper(wait)
        self.last_start = self.clock()

    @staticmethod
    def _retry_after(value):
        if not value:
            return 0.0
        try:
            return max(0.0, float(value))
        except ValueError:
            try:
                return max(0.0, (parsedate_to_datetime(value) - datetime.now(timezone.utc)).total_seconds())
            except (TypeError, ValueError, OverflowError):
                return 0.0

    def fetch(self, arxiv_id, version, destination, progress=None):
        """Stream one version into a temporary file, then replace destination.

        progress(done_bytes, total_bytes) is called while streaming (total is None when the
        server sends no Content-Length); it drives the "MB left" bar of download_sources.
        """
        if not ID_PATTERN.fullmatch(arxiv_id) or type(version) is not int or version < 1:
            raise ValueError("Invalid arXiv ID/version")
        destination = Path(destination)
        destination.parent.mkdir(parents=True, exist_ok=True)
        url = f"https://export.arxiv.org/src/{arxiv_id}v{version}"
        for attempt in range(self.max_attempts):
            self._pace()
            temporary = destination.with_name(destination.name + "." + uuid.uuid4().hex + ".part")
            response = None
            try:
                response = self.session.get(url, stream=True, timeout=(10, 120), allow_redirects=False)
                status = response.status_code
                if status != 200:
                    error = requests.HTTPError(f"HTTP {status} from {url}", response=response)
                    if status in RETRY_STATUSES:
                        self.not_before = max(self.not_before,
                                              self.clock() + max(2 ** attempt, self._retry_after(response.headers.get("Retry-After"))))
                    if status not in RETRY_STATUSES or attempt + 1 == self.max_attempts:
                        raise error
                    continue
                # The raw archive is a throwaway file on local runtime disk:
                # no fsync, it is extracted and deleted right after.
                size = 0
                length = response.headers.get("Content-Length", "")
                total = int(length) if length.isdigit() else None
                if progress:
                    progress(0, total)
                with temporary.open("wb") as output:
                    for block in response.iter_content(chunk_size=1024 * 1024):
                        if block:
                            output.write(block)
                            size += len(block)
                            if progress:
                                progress(size, total)
                if not size:
                    raise requests.RequestException(f"Empty source response from {url}")
                os.replace(temporary, destination)
                return size
            except requests.HTTPError:
                raise
            except (requests.RequestException, OSError):
                if attempt + 1 == self.max_attempts:
                    raise
                self.sleeper(2 ** attempt)
            finally:
                if response is not None:
                    response.close()
                temporary.unlink(missing_ok=True)
        raise RuntimeError("Unreachable retry state")

## 4. Keep only `.tex` and `.bib`

`extract_source` accepts TAR (any compression), ZIP, a single gzip/bzip2/xz file, plain TeX, or a
PDF/Word file (no source). Every archive path is checked before anything is written. `before_bytes`
is the unpacked size of all files; `after_bytes` is the size of the kept `.tex`/`.bib` files.

In [ ]:
def _safe_name(name):
    raw = str(name).replace("\\", "/")
    if (not raw or raw.startswith("/") or "\x00" in raw or ":" in raw
            or any(part in ("..", "") for part in raw.split("/"))):
        raise ValueError(f"Unsafe archive path: {name!r}")
    return "/".join(part for part in raw.split("/") if part != ".")


def _check_members(entries):
    """Reject unsafe paths, and a file whose path is also used as a folder (exact names)."""
    files, folders = set(), set()
    for name, is_file in entries:
        normalized = _safe_name(name)
        if not normalized:
            if is_file:
                raise ValueError("Archive file has no path")
            continue
        if is_file:
            files.add(normalized)
            folders.update(str(p) for p in PurePosixPath(normalized).parents if str(p) != ".")
    conflicts = files & folders
    if conflicts:
        raise ValueError(f"Archive path conflicts with file: {sorted(conflicts)[0]!r}")


def _case_insensitive(directory):
    """True if `directory` treats A.tex and a.tex as one file (Windows, macOS)."""
    directory = Path(directory)
    directory.mkdir(parents=True, exist_ok=True)
    probe = directory / f".case-probe-{uuid.uuid4().hex}"
    probe.touch()
    try:
        return probe.with_name(probe.name.upper()).exists()
    finally:
        probe.unlink(missing_ok=True)


def _store_name(name, taken, fold_case):
    """Where to store a kept file: A.tex/a.tex clashes are renamed only on case-insensitive disks."""
    if not fold_case or name.casefold() not in taken:
        return name
    path, n = PurePosixPath(name), 2
    while str(path.with_name(f"{path.stem}~{n}{path.suffix}")).casefold() in taken:
        n += 1
    return str(path.with_name(f"{path.stem}~{n}{path.suffix}"))


def _copy_text(stream, destination):
    destination.parent.mkdir(parents=True, exist_ok=True)
    digest, size = hashlib.sha256(), 0
    with destination.open("xb") as output:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            size += len(block)
            if size > MAX_TEXT_BYTES:
                raise ValueError("Source text file exceeds safety limit")
            digest.update(block)
            output.write(block)
    return {"size": size, "sha256": digest.hexdigest()}


def _known_non_tex(header):
    return header.startswith((b"%PDF-", b"\xd0\xcf\x11\xe0"))


def _looks_like_markup(header):
    return header.lstrip().lower().startswith((b"<html", b"<!doctype", b"<?xml", b"{"))


def _kept(name):
    return PurePosixPath(name).suffix.lower() in KEEP


def _single_tex(stream, first, destination):
    """Stream a single-file source to main.tex in one pass, checking it is TeX."""
    target = destination / "main.tex"
    digest, size, found, tail = hashlib.sha256(), 0, False, b""
    try:
        with target.open("xb") as output:
            block = first
            while block:
                size += len(block)
                if size > MAX_TEXT_BYTES:
                    raise ValueError("Single-file source exceeds safety limit")
                found = found or bool(TEX_COMMAND.search(tail + block))
                tail = block[-64:]
                digest.update(block)
                output.write(block)
                block = stream.read(1024 * 1024)
        if not found:
            raise ValueError("Unrecognized single-file source response")
    except BaseException:
        target.unlink(missing_ok=True)
        raise
    return size, {"main.tex": {"size": size, "sha256": digest.hexdigest()}}


def _extract_members(members, open_member, destination, result, fold_case):
    """Check every path, then copy .tex/.bib only if the archive has a .tex.

    A name listed twice keeps its last copy, as tar and unzip do. Names that differ only in case
    (2609.32867v1 has sections/limitations.tex and sections/Limitations.tex, with different text)
    are different files and both are kept. On a case-insensitive disk the later one is stored as
    Limitations~2.tex and result["renamed"] maps the stored name back to the original one.
    """
    _check_members([(name.rstrip("/"), is_file) for name, is_file, _size, _m in members])
    files = {}
    for name, is_file, size, member in members:
        if is_file:
            files[_safe_name(name)] = (size, member)   # a repeated name keeps the last copy
    result["before_bytes"] = sum(size for size, _member in files.values())
    if not any(PurePosixPath(name).suffix.lower() == ".tex" for name in files):
        return
    taken = set()
    for name, (_size, member) in files.items():
        if not _kept(name):
            continue
        stored = _store_name(name, taken, fold_case)
        taken.add(stored.casefold())
        if stored != name:
            result.setdefault("renamed", {})[stored] = name
        with open_member(member) as stream:
            result["files"][stored] = _copy_text(stream, destination / stored)


def _extract_zip(archive, destination, result, fold_case):
    with zipfile.ZipFile(archive) as package:
        members = package.infolist()
        for member in members:
            mode = (member.external_attr >> 16) & 0o170000
            if mode not in (0, 0o100000, 0o040000):
                raise ValueError(f"Special ZIP member: {member.filename!r}")
        _extract_members([(m.filename, not m.is_dir(), m.file_size, m) for m in members],
                         package.open, destination, result, fold_case)


def _open_tar(archive):
    try:
        return tarfile.open(archive, mode="r:*")
    except tarfile.TarError:
        return None


def _extract_tar(package, destination, result, fold_case):
    with package:
        members = package.getmembers()
        for member in members:
            if not (member.isfile() or member.isdir()):
                raise ValueError(f"Special TAR member: {member.name!r}")
        _extract_members([(m.name, m.isfile(), m.size, m) for m in members],
                         package.extractfile, destination, result, fold_case)


def extract_source(archive, destination, fold_case=None):
    """Write only .tex/.bib; compute uncompressed member sizes without storing figures.

    fold_case: rename A.tex/a.tex clashes (default: only if `destination` is case-insensitive).
    """
    archive, destination = Path(archive), Path(destination)
    destination.mkdir(parents=True, exist_ok=True)
    if fold_case is None:
        fold_case = _case_insensitive(destination)
    result = {"status": "empty_source", "before_bytes": 0, "after_bytes": 0, "files": {}}
    with archive.open("rb") as stream:
        header = stream.read(8192)
    if zipfile.is_zipfile(archive):
        _extract_zip(archive, destination, result, fold_case)
    elif (package := _open_tar(archive)) is not None:
        _extract_tar(package, destination, result, fold_case)
    else:
        opener = gzip.open if header.startswith(b"\x1f\x8b") else (
            bz2.open if header.startswith(b"BZh") else (
                lzma.open if header.startswith(b"\xfd7zXZ") else None))
        if opener is not None:
            with opener(archive, "rb") as stream:
                first = stream.read(8192)
                if _known_non_tex(first):
                    result["before_bytes"] = len(first) + sum(len(block) for block in iter(lambda: stream.read(1024 * 1024), b""))
                elif _looks_like_markup(first):
                    raise ValueError("Unrecognized HTML/JSON source response")
                else:
                    result["before_bytes"], result["files"] = _single_tex(stream, first, destination)
        elif _known_non_tex(header):
            result["before_bytes"] = archive.stat().st_size
        elif _looks_like_markup(header):
            raise ValueError("Unrecognized HTML/JSON source response")
        else:
            with archive.open("rb") as stream:
                result["before_bytes"], result["files"] = _single_tex(stream, stream.read(1024 * 1024), destination)
    if any(PurePosixPath(name).suffix.lower() == ".tex" for name in result["files"]):
        result["status"] = "ok"
        result["after_bytes"] = sum(entry["size"] for entry in result["files"].values())
    return result

## 5. Receipts and fast resume

A version counts as done only if its receipt matches the files on disk. The `receipts.jsonl` index is
read in one pass; the per-version receipt file is the fallback (e.g. after a crash between the two writes).
Only versions that have a receipt are checked, so the startup check grows with the versions already
downloaded, not with every assigned ID (each lookup on Drive takes ~10-30 ms).

In [ ]:
def _receipt_path(state_dir, arxiv_id, version):
    return Path(state_dir) / "receipts" / to_folder_id(arxiv_id) / f"v{version}.json"


def _index_path(state_dir):
    return Path(state_dir) / "receipts.jsonl"


def _read_receipt_index(state_dir):
    """One sequential read instead of thousands of small receipt files on Drive."""
    index = {}
    try:
        with _index_path(state_dir).open(encoding="utf-8") as stream:
            for line in stream:
                try:
                    receipt = json.loads(line)
                    index[(receipt["arxiv_id"], receipt["version"])] = receipt
                except (ValueError, KeyError, TypeError):
                    continue  # A torn final line after a crash is ignored.
    except FileNotFoundError:
        pass
    return index


def _file_digest(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def _check_receipt(receipt, directory, arxiv_id, version, verify):
    """Compare a receipt with the files on disk (sizes; SHA-256 if verify='sha256')."""
    if (receipt["arxiv_id"] != arxiv_id or receipt["version"] != version
            or receipt["status"] not in ("ok", "empty_source")):
        return False
    files = receipt["files"]
    if receipt["status"] == "empty_source" and files:
        return False
    if receipt["status"] == "ok" and not any(PurePosixPath(n).suffix.lower() == ".tex" for n in files):
        return False
    actual = {}
    for root, _dirs, names in os.walk(directory):
        for name in names:
            path = Path(root) / name
            if path.is_symlink():
                return False
            actual[path.relative_to(directory).as_posix()] = path.stat().st_size
    if actual != {name: entry["size"] for name, entry in files.items()}:
        return False
    if verify == "sha256":
        return all(_file_digest(directory / name) == entry["sha256"] for name, entry in files.items())
    return True


def _prune_unkept(receipt, directory, state_dir):
    """Drop files kept by older runs (.bbl/.sty/.cls) without re-downloading."""
    extra = [name for name in receipt["files"] if not _kept(name)]
    if not extra:
        return receipt
    for name in extra:
        path = directory / name
        path.unlink(missing_ok=True)
        for parent in path.parents:
            if parent == directory or not parent.is_relative_to(directory):
                break
            try:
                parent.rmdir()
            except OSError:
                break
    files = {name: entry for name, entry in receipt["files"].items() if _kept(name)}
    receipt = dict(receipt, files=files, after_bytes=sum(entry["size"] for entry in files.values()))
    _atomic_json(_receipt_path(state_dir, receipt["arxiv_id"], receipt["version"]), receipt)
    return receipt


def verified_receipt(state_dir, output_dir, arxiv_id, version, *, verify="size", cached=None):
    """Return a receipt that matches the saved files, or None to re-download.

    ``cached`` (from receipts.jsonl) is tried first; the per-version receipt
    file is the fallback, e.g. after a crash between the two writes.
    """
    directory = Path(output_dir) / to_folder_id(arxiv_id) / f"v{version}"
    try:
        if not directory.is_dir() or directory.is_symlink():
            return None
    except OSError:
        return None
    for receipt in ([cached] if cached is not None else []) + [None]:
        if receipt is None:  # Fallback: the per-version receipt file.
            try:
                receipt = json.loads(_receipt_path(state_dir, arxiv_id, version).read_text(encoding="utf-8"))
            except (OSError, ValueError):
                return None
        try:
            if _check_receipt(receipt, directory, arxiv_id, version, verify):
                return _prune_unkept(receipt, directory, state_dir)
        except (OSError, ValueError, KeyError, TypeError, AttributeError):
            pass
    return None


def _ids_with_receipt_files(state_dir):
    """Folder names in state/receipts/ (one listing): the papers that have per-version receipt files."""
    try:
        with os.scandir(Path(state_dir) / "receipts") as entries:
            return {entry.name for entry in entries}
    except FileNotFoundError:
        return set()


def load_receipts(records, output_dir, state_dir, *, verify="size"):
    """Verify saved versions once at startup; refresh the index for pruned receipts.

    Only versions with a receipt (receipts.jsonl or a per-version file) can pass, so the others are
    skipped without a lookup in output_dir: on Drive each lookup costs ~10-30 ms, for every assigned version.
    """
    index = _read_receipt_index(state_dir)
    indexed = {arxiv_id for arxiv_id, _version in index}
    with_files = _ids_with_receipt_files(state_dir)
    receipts, changed = {}, []
    for record in records:
        if record["arxiv_id"] not in indexed and to_folder_id(record["arxiv_id"]) not in with_files:
            continue
        for version in range(1, record["version_count"] + 1):
            key = (record["arxiv_id"], version)
            receipt = verified_receipt(state_dir, output_dir, *key, verify=verify, cached=index.get(key))
            if receipt is not None:
                receipts[key] = receipt
                if receipt is not index.get(key):
                    changed.append(receipt)
    if changed:
        with _index_path(state_dir).open("a", encoding="utf-8") as stream:
            stream.writelines(json.dumps(r, ensure_ascii=False) + "\n" for r in changed)
    return receipts

## 6. Pipelined download and reporting

`process_download` runs in a worker thread: extract on local disk, copy the kept files into a Drive
staging folder, rename it to `vN/` and write the receipt. `download_sources` writes `progress.json`,
`run_summary.json`, `run_history.jsonl`, `errors.jsonl` and `papers.csv` (columns `LOG_FIELDS` from
`00_config`, as read by `04_monitor.ipynb`).

In [ ]:
_OUTPUT_CASE_INSENSITIVE = {}   # output folder -> bool, probed once (Drive calls are slow)


def _output_case_insensitive(output_dir):
    key = str(Path(output_dir).resolve())
    if key not in _OUTPUT_CASE_INSENSITIVE:
        _OUTPUT_CASE_INSENSITIVE[key] = _case_insensitive(output_dir)
    return _OUTPUT_CASE_INSENSITIVE[key]


def process_download(raw, arxiv_id, version, downloaded, output_dir, state_dir, started=None):
    """Extract a fetched archive locally, then publish only kept files to output_dir."""
    started = time.perf_counter() if started is None else started
    raw, output_dir, state_dir = Path(raw), Path(output_dir), Path(state_dir)
    with tempfile.TemporaryDirectory(dir=raw.parent) as local:
        # Extract on fast local disk; rejected archives never touch Drive.
        fold_case = _case_insensitive(local) or _output_case_insensitive(output_dir)
        info = extract_source(raw, Path(local) / "source", fold_case=fold_case)
        raw.unlink(missing_ok=True)  # Free local disk before the Drive copy.
        stage = output_dir / ".staging"
        stage.mkdir(parents=True, exist_ok=True)
        with tempfile.TemporaryDirectory(dir=stage) as temporary:
            extracted = Path(temporary) / "source"
            extracted.mkdir()
            for name in info["files"]:
                destination = extracted / name
                destination.parent.mkdir(parents=True, exist_ok=True)
                shutil.copyfile(Path(local) / "source" / name, destination)
            target = output_dir / to_folder_id(arxiv_id) / f"v{version}"
            target.parent.mkdir(parents=True, exist_ok=True)
            backup = None
            if target.exists():
                backup = state_dir / "replaced" / f"{to_folder_id(arxiv_id)}_v{version}_{uuid.uuid4().hex}"
                backup.parent.mkdir(parents=True, exist_ok=True)
                os.replace(target, backup)
            try:
                os.replace(extracted, target)
                receipt = dict(info, arxiv_id=arxiv_id, version=version,
                               downloaded_bytes=downloaded, finished_at=_now(),
                               elapsed_seconds=round(time.perf_counter() - started, 3))
                _atomic_json(_receipt_path(state_dir, arxiv_id, version), receipt)
            except Exception:
                if backup is not None and not target.exists():
                    os.replace(backup, target)
                raise
            if backup is not None:
                shutil.rmtree(backup)
            return receipt


def collect_one(client, record, version, output_dir, state_dir):
    """Fetch and process one version synchronously (used by tests and smoke runs)."""
    started = time.perf_counter()
    with tempfile.TemporaryDirectory() as local_temporary:
        raw = Path(local_temporary) / "download.bin"
        downloaded = client.fetch(record["arxiv_id"], version, raw)
        return process_download(raw, record["arxiv_id"], version, downloaded,
                                output_dir, state_dir, started)


class _MemorySampler:
    def __init__(self):
        self.values = []
        self.stop = threading.Event()
        self.thread = threading.Thread(target=self._sample, daemon=True)

    def _sample(self):
        process = psutil.Process() if psutil else None
        while not self.stop.is_set():
            if process:
                self.values.append(process.memory_info().rss)
            self.stop.wait(2)

    def __enter__(self):
        self.thread.start()
        return self

    def __exit__(self, *_):
        self.stop.set()
        self.thread.join()


def _progress(records, receipts):
    total_papers = len(records)
    total_versions = sum(r["version_count"] for r in records)
    completed_versions = len(receipts)
    completed_papers = sum(
        (r["version_count"] == 0 and r.get("status") == "not_found") or
        (r["version_count"] > 0 and all((r["arxiv_id"], v) in receipts for v in range(1, r["version_count"] + 1)))
        for r in records
    )
    return {"papers_completed": completed_papers, "papers_total": total_papers,
            "papers_percent": round(100 * completed_papers / total_papers, 2),
            "versions_completed": completed_versions, "versions_total": total_versions,
            "versions_percent": round(100 * completed_versions / total_versions, 2) if total_versions else 0.0}


def _print_progress(progress, say=print):
    say(f"Papers: {progress['papers_completed']:,}/{progress['papers_total']:,} ({progress['papers_percent']:.2f}%) | "
          f"versions: {progress['versions_completed']:,}/{progress['versions_total']:,} ({progress['versions_percent']:.2f}%)")


def _write_papers_csv(path, records, receipts):
    """Export downloader metrics in the column schema of 04_monitor.ipynb."""
    path = Path(path)
    temporary = path.with_name(path.name + "." + uuid.uuid4().hex + ".tmp")
    try:
        with temporary.open("w", newline="", encoding="utf-8") as stream:
            writer = csv.DictWriter(stream, fieldnames=LOG_FIELDS)
            writer.writeheader()
            for record in records:
                identifier, count = record["arxiv_id"], record["version_count"]
                found = [receipts[(identifier, version)] for version in range(1, count + 1)
                         if (identifier, version) in receipts]
                status = (record.get("status", "error") if count == 0 else
                          "ok" if len(found) == count and all(r["status"] == "ok" for r in found) else
                          "partial" if found else "error")
                writer.writerow({"id": identifier, "versions": count,
                                 "versions_ok": sum(r["status"] == "ok" for r in found),
                                 "size_before": sum(r["before_bytes"] for r in found),
                                 "size_after": sum(r["after_bytes"] for r in found),
                                 "status": status,
                                 "seconds": round(sum(r.get("elapsed_seconds", 0) for r in found), 3),
                                 "error": "unprocessed source versions" if len(found) < count else "",
                                 "logged_at": _now()})
            stream.flush()
            os.fsync(stream.fileno())
        os.replace(temporary, path)
    finally:
        temporary.unlink(missing_ok=True)


def _process_job(job_dir, *args):
    try:
        return process_download(Path(job_dir) / "download.bin", *args)
    finally:
        shutil.rmtree(job_dir, ignore_errors=True)


_FATAL_ERRNOS = {code for code in (errno.ENOSPC, getattr(errno, "EDQUOT", None), errno.ENOTCONN, errno.EROFS) if code}


def _is_fatal(exc):
    """A storage error that every later version would hit too: disk or Drive quota full, Drive unmounted."""
    for _ in range(10):   # follow the chain of causes
        if exc is None:
            return False
        if isinstance(exc, OSError) and exc.errno in _FATAL_ERRNOS:
            return True
        exc = exc.__cause__ or exc.__context__
    return False


class RunLock:
    """Refuse to run while another runtime works on the same folder (best effort, also on Drive).

    The lock file holds the owner (host:pid) and a heartbeat refreshed every `beat` seconds. It is taken
    over when the heartbeat is older than `stale` seconds (the other Colab runtime is gone) or, on the same
    host, when the owner's process no longer exists (a restarted kernel). Re-entrant within one process.
    """

    _held = {}   # lock path -> [depth, stop event, heartbeat thread]

    def __init__(self, path, beat=60, stale=180):
        self.path, self.beat, self.stale = Path(path), beat, stale
        self.owner = f"{socket.gethostname()}:{os.getpid()}"

    def _read(self):
        try:
            return json.loads(self.path.read_text(encoding="utf-8"))
        except (OSError, ValueError):
            return None

    def _write(self):
        _atomic_json(self.path, {"owner": self.owner, "heartbeat": time.time(), "heartbeat_utc": _now()})

    def acquire(self):
        key = str(self.path.resolve())
        if key in RunLock._held:
            RunLock._held[key][0] += 1
            return self
        other = self._read()
        if other and other.get("owner") != self.owner:
            age = time.time() - float(other.get("heartbeat", 0))
            host, _, pid = str(other.get("owner", "")).rpartition(":")
            gone = (host == socket.gethostname() and pid.isdigit()
                    and psutil is not None and not psutil.pid_exists(int(pid)))
            if age < self.stale and not gone:
                raise RuntimeError(
                    f"Another run ({other.get('owner')}) is working on {self.path.parent} "
                    f"(last heartbeat {age:.0f} s ago). Stop it first. If it was a Colab runtime that no longer "
                    f"exists, run again in {self.stale - age:.0f} s, or delete {self.path}.")
        self.path.parent.mkdir(parents=True, exist_ok=True)
        self._write()
        stop = threading.Event()

        def heartbeat():
            while not stop.wait(self.beat):
                try:
                    self._write()
                except OSError:
                    pass

        thread = threading.Thread(target=heartbeat, daemon=True)
        thread.start()
        RunLock._held[key] = [1, stop, thread]
        return self

    def release(self):
        key = str(self.path.resolve())
        held = RunLock._held.get(key)
        if not held:
            return
        held[0] -= 1
        if held[0] > 0:
            return
        held[1].set()
        held[2].join(timeout=5)
        del RunLock._held[key]
        if (self._read() or {}).get("owner") == self.owner:
            self.path.unlink(missing_ok=True)

    def __enter__(self):
        return self.acquire()

    def __exit__(self, *exc_info):
        self.release()


def _free_bytes(folder):
    try:
        return shutil.disk_usage(folder).free
    except OSError:
        return None


def check_free_space(records, receipts, output_dir, temp_dir, workers):
    """Problems (empty list if none) if the output or temp folder cannot hold the rest of the crawl."""
    mb = 1024 * 1024
    remaining = sum(1 for r in records for v in range(1, r["version_count"] + 1) if (r["arxiv_id"], v) not in receipts)
    kept = [r["after_bytes"] for r in receipts.values() if r["status"] == "ok"]
    per_version = sum(kept) / len(kept) if kept else 300 * 1024
    needs = ((f"output folder {output_dir}", output_dir, int(remaining * per_version * 1.5) + 50 * mb),
             (f"local temp folder {temp_dir}", temp_dir, 2 * workers * 200 * mb))
    problems = []
    for label, folder, need in needs:
        free = _free_bytes(folder)
        if remaining and free is not None and free < need:
            problems.append(f"{label}: {free / mb:,.0f} MB free, about {need / mb:,.0f} MB needed")
    return problems


def _clean_leftovers(output_dir, state_dir):
    """Staging folders and backups left by a killed run (safe: RunLock keeps other downloaders out)."""
    removed = 0
    for folder in (Path(output_dir) / ".staging", Path(state_dir) / "replaced"):
        if folder.is_dir():
            for child in folder.iterdir():
                if child.is_dir():
                    shutil.rmtree(child, ignore_errors=True)
                else:
                    child.unlink(missing_ok=True)
                removed += 1
    return removed


def download_sources(manifest_path, output_dir, state_dir, *, user_agent=USER_AGENT,
        delay=ARXIV_DELAY, session=None, sleeper=time.sleep, workers=2, verify="size",
        temp_dir=None, progress=False, max_consecutive_failures=20, check_space=True):
    """Run or resume the downloader; errors are logged and retried next run.

    One paced connection fetches archives to local temp storage while up to
    ``workers`` threads extract them and write .tex/.bib to ``output_dir``.
    ``verify='sha256'`` re-hashes saved files at startup (slower on Drive).
    ``progress=True`` shows two bars: papers done out of every discovered paper (with versions,
    MB downloaded and ETA), and the file being downloaded with the MB left.

    Safety: a lock keeps a second runtime out of ``state_dir``; leftovers of a killed run are removed;
    the free space is checked before the first request (``check_space``); and the run stops early
    (``stopped_reason`` in the summary) on a storage error (disk/Drive full, Drive unmounted) or after
    ``max_consecutive_failures`` failed versions in a row, instead of failing every remaining version.
    """
    if workers < 1:
        raise ValueError("workers must be positive")
    if verify not in ("size", "sha256"):
        raise ValueError("verify must be 'size' or 'sha256'")
    start = time.perf_counter()
    records = load_manifest(manifest_path)
    output_dir, state_dir = Path(output_dir), Path(state_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    state_dir.mkdir(parents=True, exist_ok=True)
    lock = RunLock(state_dir / "download.lock").acquire()
    try:
        return _download_locked(records, output_dir, state_dir, start, user_agent, delay, session, sleeper,
                                workers, verify, temp_dir, progress, max_consecutive_failures, check_space)
    finally:
        lock.release()


def _download_locked(records, output_dir, state_dir, start, user_agent, delay, session, sleeper,
                     workers, verify, temp_dir, progress, max_consecutive_failures, check_space):
    removed = _clean_leftovers(output_dir, state_dir)
    if removed:
        print(f"Removed {removed} leftover staging folder(s)/backup(s) of an interrupted run")
    receipts = load_receipts(records, output_dir, state_dir, verify=verify)
    print(f"Progress at start/resume (verified in {time.perf_counter() - start:.1f}s):")
    _print_progress(_progress(records, receipts))
    if check_space:
        problems = check_free_space(records, receipts, output_dir, temp_dir or tempfile.gettempdir(), workers)
        if problems:
            raise RuntimeError("Not enough free space; free some space (or pass check_space=False):\n  "
                               + "\n  ".join(problems))
    skipped = failed = consecutive = 0
    stop_reason = None
    current_bytes = sum(r["after_bytes"] for r in receipts.values())
    disk_peak_estimate = current_bytes
    client = SourceClient(session=session, user_agent=user_agent, delay=delay, sleeper=sleeper)
    owns_session = session is None
    interrupted = False
    max_pending = 2 * workers  # Bounds local disk used by fetched, unprocessed archives.
    pending = {}
    pending_bytes = {}          # future -> size of its raw archive, still on local disk
    local_bytes = local_peak = 0
    work_root = Path(tempfile.mkdtemp(prefix="arxiv-src-", dir=temp_dir))
    index_stream = _index_path(state_dir).open("a", encoding="utf-8")

    counts = {r["arxiv_id"]: r["version_count"] for r in records}
    versions_total = sum(counts.values())
    completed = {pid for pid, n in counts.items()
                 if n > 0 and all((pid, v) in receipts for v in range(1, n + 1))}
    session_bytes = 0
    paper_bar = file_bar = None
    if progress and tqdm is not None:
        paper_bar = tqdm(total=sum(n > 0 for n in counts.values()), initial=len(completed),
                         unit="paper", desc="Papers", position=0, dynamic_ncols=True)
        file_bar = tqdm(total=0, unit="B", unit_scale=True, unit_divisor=1024, desc="Next file",
                        position=1, leave=False, dynamic_ncols=True)
    say = tqdm.write if paper_bar is not None else print

    def refresh_bars():
        if paper_bar is not None:
            paper_bar.set_postfix_str(f"versions {len(receipts):,}/{versions_total:,} | "
                                      f"{session_bytes / 1048576:,.0f} MB downloaded")

    def file_progress(label):
        """Callback for SourceClient.fetch: the current file, its size and the MB left."""
        def update(done, total):
            if done == 0:
                file_bar.reset(total=total)
                file_bar.set_description(label, refresh=False)
            if total:   # set before update(), so the bar is drawn with the current value
                file_bar.set_postfix_str(f"{(total - done) / 1048576:,.1f} MB left", refresh=False)
            file_bar.update(done - file_bar.n)
        return update

    def stop_for_storage(exc):
        nonlocal stop_reason
        if stop_reason is None:
            stop_reason = f"storage error, every later version would fail too: {exc!r}"

    def log_failure(identifier, version, exc):
        nonlocal failed, consecutive, stop_reason
        failed += 1
        consecutive += 1
        try:
            with (state_dir / "errors.jsonl").open("a", encoding="utf-8") as errors:
                errors.write(json.dumps({"arxiv_id": identifier, "version": version,
                                         "error": repr(exc), "at": _now()}, ensure_ascii=False) + "\n")
        except OSError as write_error:
            stop_for_storage(write_error)
        say(f"FAILED {identifier}v{version}: {exc}")
        if _is_fatal(exc):
            stop_for_storage(exc)
        elif stop_reason is None and consecutive >= max_consecutive_failures:
            stop_reason = (f"{consecutive} versions failed in a row (last: {exc!r}); arXiv, the network "
                           f"or Google Drive is probably down")

    def harvest(block):
        nonlocal current_bytes, disk_peak_estimate, local_bytes, consecutive
        if not pending:
            return
        done = wait(pending, return_when=FIRST_COMPLETED)[0] if block else [f for f in pending if f.done()]
        for future in done:
            key = pending.pop(future)
            local_bytes -= pending_bytes.pop(future, 0)
            try:
                receipt = future.result()
            except Exception as exc:
                log_failure(*key, exc)
                continue
            consecutive = 0
            disk_peak_estimate = max(disk_peak_estimate,
                                     current_bytes + receipt["downloaded_bytes"] + receipt["after_bytes"])
            current_bytes += receipt["after_bytes"]
            receipts[key] = receipt
            try:
                index_stream.write(json.dumps(receipt, ensure_ascii=False) + "\n")
                index_stream.flush()
            except OSError as exc:
                stop_for_storage(exc)
            pid = key[0]
            if pid not in completed and all((pid, v) in receipts for v in range(1, counts[pid] + 1)):
                completed.add(pid)
                if paper_bar is not None:
                    paper_bar.update(1)
            refresh_bars()

    try:
        with _MemorySampler() as memory, ThreadPoolExecutor(max_workers=workers) as pool:
            try:
                for index, record in enumerate(records, 1):
                    identifier = record["arxiv_id"]
                    for version in range(1, record["version_count"] + 1):
                        if stop_reason:
                            break
                        key = (identifier, version)
                        if key in receipts:
                            skipped += 1
                            continue
                        harvest(block=False)
                        while len(pending) >= max_pending:
                            harvest(block=True)
                        if stop_reason:
                            break
                        started = time.perf_counter()
                        job = Path(tempfile.mkdtemp(dir=work_root))
                        try:
                            downloaded = client.fetch(
                                identifier, version, job / "download.bin",
                                progress=file_progress(f"{identifier}v{version}") if file_bar is not None else None)
                        except Exception as exc:
                            shutil.rmtree(job, ignore_errors=True)
                            log_failure(identifier, version, exc)
                            continue
                        except BaseException:
                            shutil.rmtree(job, ignore_errors=True)
                            raise
                        future = pool.submit(_process_job, job, identifier, version, downloaded,
                                             output_dir, state_dir, started)
                        pending[future] = key
                        pending_bytes[future] = downloaded
                        local_bytes += downloaded
                        local_peak = max(local_peak, local_bytes)
                        session_bytes += downloaded
                        refresh_bars()
                    if stop_reason:
                        break
                    if index % 100 == 0 or index == len(records):
                        harvest(block=False)
                        current = _progress(records, receipts)
                        _print_progress(current, say)
                        try:
                            _atomic_json(state_dir / "progress.json", current)
                        except OSError as exc:
                            stop_for_storage(exc)
            except KeyboardInterrupt:
                interrupted = True
                say("Interrupted; finishing archives already downloaded, then saving receipts.")
            finally:
                while pending:  # Archives already fetched are never thrown away.
                    harvest(block=True)
    finally:
        for bar in (file_bar, paper_bar):
            if bar is not None:
                bar.close()
        try:
            index_stream.close()
        except OSError:
            pass
        shutil.rmtree(work_root, ignore_errors=True)
        if owns_session:
            client.session.close()
    progress = _progress(records, receipts)
    final_bytes = sum(r["after_bytes"] for r in receipts.values())
    summary = {"finished_at": _now(), "elapsed_seconds": round(time.perf_counter() - start, 3),
               "interrupted": interrupted, "stopped_reason": stop_reason, "progress": progress,
               "workers": workers, "verify": verify, "kept_extensions": sorted(KEEP),
               "elapsed_per_assigned_paper_seconds": round((time.perf_counter() - start) / len(records), 6),
               "versions_ok": sum(r["status"] == "ok" for r in receipts.values()),
               "versions_empty": sum(r["status"] == "empty_source" for r in receipts.values()),
               "papers_all_sources": sum(
                   r["version_count"] > 0 and all(
                       (r["arxiv_id"], v) in receipts and receipts[(r["arxiv_id"], v)]["status"] == "ok"
                       for v in range(1, r["version_count"] + 1)) for r in records),
               "versions_skipped": skipped, "versions_failed_this_run": failed,
               "source_before_bytes": sum(r["before_bytes"] for r in receipts.values()),
               "source_after_bytes": final_bytes,
               "output_bytes": final_bytes,
               "disk_peak_estimate_bytes": max(disk_peak_estimate, final_bytes),
               "local_temp_peak_bytes": local_peak,   # raw archives waiting on local disk at once
               "ram_peak_bytes": max(memory.values, default=None),
               "ram_average_bytes": round(sum(memory.values) / len(memory.values)) if memory.values else None}
    try:
        _atomic_json(state_dir / "run_summary.json", summary)
        _atomic_json(state_dir / "progress.json", progress)
        _write_papers_csv(state_dir / "papers.csv", records, receipts)
        with (state_dir / "run_history.jsonl").open("a", encoding="utf-8") as history:
            history.write(json.dumps(summary, ensure_ascii=False) + "\n")
            history.flush()
            os.fsync(history.fileno())
    except OSError as exc:
        print(f"Could not write the run summary: {exc!r}")
    _print_progress(progress)
    if stop_reason:
        print(f"\nSTOPPED EARLY: {stop_reason}\nFix the cause (free space on Drive, re-mount Drive, wait for "
              f"arXiv), then run again: finished versions are kept and the failed ones are retried.")
    return summary

## 7. Offline tests

These cells check the downloader with temporary files and fake HTTP sessions; they do not contact
arXiv. The test classes are always defined but run only when `RUN_TEST` is `True` (see `00_config`).

In [ ]:
"""Offline contract tests for the member-2 source downloader."""

import gzip
import csv
import io
import json
from pathlib import Path
import sys
import tarfile
import tempfile
import unittest
from unittest.mock import patch
import zipfile



TEX = b"\\documentclass{article}\n\\begin{figure}\\includegraphics{plot}\n"


def tar_bytes(files):
    out = io.BytesIO()
    with tarfile.open(fileobj=out, mode="w:gz") as archive:
        for name, content in files:
            info = tarfile.TarInfo(name)
            info.size = len(content)
            archive.addfile(info, io.BytesIO(content))
    return out.getvalue()


class FakeResponse:
    def __init__(self, content, status=200, headers=None):
        self.content = content
        self.status_code = status
        self.headers = headers or {}

    def iter_content(self, chunk_size):
        yield self.content

    def close(self):
        pass


class FakeSession:
    def __init__(self, responses):
        self.responses = iter(responses)
        self.calls = []
        self.headers = {}

    def get(self, url, **kwargs):
        self.calls.append((url, kwargs))
        return next(self.responses)

    def close(self):
        pass



class DownloadTestBase(unittest.TestCase):
    def setUp(self):
        self.temp = tempfile.TemporaryDirectory()
        self.addCleanup(self.temp.cleanup)
        self.root = Path(self.temp.name)


### 7a. Manifest and request pacing

Validate discovery records, assigned ranges, retries, and the 3-second request gate.

In [ ]:
class ManifestAndPacingTests(DownloadTestBase):
    def test_manifest_accepts_group_discovery_schema(self):
        path = self.root / "manifest.json"
        path.write_text(json.dumps({"papers": [
            {"id": "2603.20580", "versions": 2, "title": "sample", "status": "ok"},
            {"id": "2603.20581", "versions": 0, "status": "not_found"},
        ]}), encoding="utf-8")
        records = load_manifest(path)
        self.assertEqual(records[0]["arxiv_id"], "2603.20580")
        self.assertEqual(records[0]["version_count"], 2)
        self.assertEqual(records[0]["title"], "sample")
        self.assertEqual(records[1]["version_count"], 0)

    def test_manifest_rejects_duplicate_and_bad_version_count(self):
        for records in (
            [{"arxiv_id": "2603.20580", "version_count": 1}] * 2,
            [{"id": "2603.20580", "versions": 0, "status": "ok"}],
            [{"arxiv_id": "2603.00000", "version_count": 1}],
        ):
            with self.subTest(records=records), self.assertRaises(ValueError):
                validate_manifest(records)

    def test_discovery_error_with_no_versions_stays_pending(self):
        records = validate_manifest([{"id": "2603.20580", "versions": 0, "status": "error"}])
        self.assertEqual(records[0]["version_count"], 0)
        self.assertEqual(_progress(records, {})["papers_percent"], 0.0)

    def test_429_retry_after_and_request_pacing(self):
        session = FakeSession([FakeResponse(b"busy", 429, {"Retry-After": "5"}), FakeResponse(b"abc")])
        current = [0.0]
        def sleep(seconds):
            current[0] += seconds
        starts = []
        original_get = session.get
        def observed_get(url, **kwargs):
            starts.append(current[0])
            return original_get(url, **kwargs)
        session.get = observed_get
        client = SourceClient(session, "CourseBot/1.0 (contact@example.org)", clock=lambda: current[0], sleeper=sleep)
        target = self.root / "raw.bin"
        self.assertEqual(client.fetch("2603.20580", 2, target), 3)
        self.assertEqual(starts, [0.0, 5.0])
        self.assertEqual(target.read_bytes(), b"abc")
        self.assertEqual([c[0] for c in session.calls], ["https://export.arxiv.org/src/2603.20580v2"] * 2)
        self.assertEqual(session.headers["User-Agent"], "CourseBot/1.0 (contact@example.org)")

    def test_final_429_cooldown_applies_to_next_version(self):
        session = FakeSession([FakeResponse(b"busy", 429, {"Retry-After": "90"}), FakeResponse(b"ok")])
        now = [0.0]
        starts = []
        original = session.get
        def get(url, **kwargs):
            starts.append(now[0])
            return original(url, **kwargs)
        session.get = get
        client = SourceClient(session, max_attempts=1, clock=lambda: now[0], sleeper=lambda s: now.__setitem__(0, now[0] + s))
        with self.assertRaises(Exception):
            client.fetch("2603.20580", 1, self.root / "raw1")
        client.fetch("2603.20580", 2, self.root / "raw2")
        self.assertEqual(starts, [0.0, 90.0])

    def test_nonretryable_404_is_sent_once(self):
        session = FakeSession([FakeResponse(b"missing", 404)])
        client = SourceClient(session, sleeper=lambda seconds: None)
        with self.assertRaises(Exception):
            client.fetch("2603.20580", 1, self.root / "raw")
        self.assertEqual(len(session.calls), 1)

    def test_assigned_manifest_coverage_rejects_truncated_cache(self):
        self.assertEqual(len(assigned_ids()), 20000)
        complete = [{"id": identifier, "versions": 1} for identifier in assigned_ids()]
        self.assertEqual(len(validate_assigned_manifest(complete)), 20000)
        with self.assertRaises(ValueError):
            validate_assigned_manifest([{"id": "2603.20580", "versions": 1}])


### 7b. Archive filtering and safety

Check exact TeX bytes, that only `.tex`/`.bib` files are written, PDF-only versions, and unsafe archive entries.

In [ ]:
class ArchiveAndSafetyTests(DownloadTestBase):
    def test_tar_keeps_tex_bytes_and_skips_figures(self):
        raw = self.root / "raw"
        raw.write_bytes(tar_bytes([("sub/main.tex", TEX), ("sub/refs.bib", b"@book{a}"), ("sub/pic.PNG", b"image"), ("sub/plot.pdf", b"%PDF")]))
        target = self.root / "v1"
        info = extract_source(raw, target)
        self.assertEqual(info["status"], "ok")
        self.assertEqual((target / "sub/main.tex").read_bytes(), TEX)
        self.assertEqual({p.suffix for p in target.rglob("*") if p.is_file()}, {".tex", ".bib"})
        self.assertEqual(info["before_bytes"], len(TEX) + len(b"@book{a}") + 5 + 4)
        self.assertEqual(info["after_bytes"], len(TEX) + len(b"@book{a}"))

    def test_gzip_single_tex_and_pdf_empty(self):
        raw = self.root / "raw"
        raw.write_bytes(gzip.compress(TEX))
        self.assertEqual(extract_source(raw, self.root / "v1")["status"], "ok")
        self.assertEqual((self.root / "v1/main.tex").read_bytes(), TEX)
        raw.write_bytes(b"%PDF-1.7\nxxx")
        info = extract_source(raw, self.root / "v2")
        self.assertEqual(info["status"], "empty_source")
        self.assertEqual(list((self.root / "v2").iterdir()), [])

    def test_single_tex_after_long_comment_is_kept(self):
        content = (b"% comment\n" * 1200) + TEX
        raw = self.root / "raw"
        for payload, folder in ((content, "plain"), (gzip.compress(content), "gz")):
            raw.write_bytes(payload)
            info = extract_source(raw, self.root / folder)
            self.assertEqual(info["status"], "ok")
            self.assertEqual((self.root / folder / "main.tex").read_bytes(), content)

    def test_unrecognized_success_body_is_not_cached_as_empty(self):
        raw = self.root / "raw"
        for payload in (b"<html>temporary gateway error</html>", b"nonsense bytes"):
            raw.write_bytes(payload)
            with self.subTest(payload=payload), self.assertRaises(ValueError):
                extract_source(raw, self.root / "v1")

    def test_single_gzip_source_is_bounded_during_decompression(self):
        raw = self.root / "raw"
        raw.write_bytes(gzip.compress(TEX + b"x" * 100))
        with (patch.dict(globals(), MAX_TEXT_BYTES=len(TEX) + 10),
              patch.object(tempfile, "TemporaryFile", side_effect=AssertionError("oversized source staged")),
              self.assertRaises(ValueError)):
            extract_source(raw, self.root / "v1")

    def test_unsafe_tar_is_rejected(self):
        raw = self.root / "raw"
        for members in ([("../escape.tex", TEX)], [("main.tex", TEX), ("main.tex/x.tex", TEX)]):
            raw.write_bytes(tar_bytes(members))
            with self.subTest(members=[m[0] for m in members]), self.assertRaises(ValueError):
                extract_source(raw, self.root / "unsafe")
        self.assertFalse((self.root / "escape.tex").exists())

    def test_case_variant_names_are_both_kept(self):
        # Real case: 2609.32867v1 has sections/limitations.tex and sections/Limitations.tex
        raw = self.root / "raw"
        raw.write_bytes(tar_bytes([("main.tex", TEX), ("sections/limitations.tex", b"lower"),
                                   ("sections/Limitations.tex", b"upper")]))
        for fold_case, folder in ((None, "auto"), (True, "folded")):
            info = extract_source(raw, self.root / folder, fold_case=fold_case)
            self.assertEqual(info["status"], "ok")
            stored = sorted((self.root / folder / name).read_bytes() for name in info["files"])
            self.assertEqual(stored, sorted([TEX, b"lower", b"upper"]))
            self.assertEqual(info["before_bytes"], len(TEX) + 10)
        self.assertEqual(info["renamed"], {"sections/Limitations~2.tex": "sections/Limitations.tex"})
        zipped = io.BytesIO()
        with zipfile.ZipFile(zipped, "w") as archive:
            archive.writestr("main.tex", TEX)
            archive.writestr("A.tex", b"upper")
            archive.writestr("a.tex", b"lower")
        raw.write_bytes(zipped.getvalue())
        info = extract_source(raw, self.root / "zip", fold_case=True)
        self.assertEqual(sorted(info["files"]), ["A.tex", "a~2.tex", "main.tex"])

    def test_repeated_tar_name_keeps_last_copy(self):
        raw = self.root / "raw"
        raw.write_bytes(tar_bytes([("main.tex", b"\\documentclass{old}"), ("main.tex", TEX)]))
        info = extract_source(raw, self.root / "v1")
        self.assertEqual((self.root / "v1/main.tex").read_bytes(), TEX)
        self.assertEqual(info["before_bytes"], len(TEX))

    def test_zip_symlink_is_rejected(self):
        out = io.BytesIO()
        with zipfile.ZipFile(out, "w") as archive:
            archive.writestr("main.tex", TEX)
            member = zipfile.ZipInfo("link.tex")
            member.create_system = 3
            member.external_attr = 0o120777 << 16
            archive.writestr(member, "main.tex")
        raw = self.root / "raw"
        raw.write_bytes(out.getvalue())
        with self.assertRaises(ValueError):
            extract_source(raw, self.root / "v1")

    def test_tar_with_dot_prefix_is_accepted(self):
        raw = self.root / "raw"
        raw.write_bytes(tar_bytes([("./main.tex", TEX), ("./fig.png", b"image")]))
        info = extract_source(raw, self.root / "v1")
        self.assertEqual(info["status"], "ok")
        self.assertEqual((self.root / "v1/main.tex").read_bytes(), TEX)

    def test_archive_download_uses_runtime_temp_not_output_drive(self):
        output, state = self.root / "output", self.root / "state"
        seen = []
        class Client:
            def fetch(self, identifier, version, destination):
                seen.append(Path(destination))
                payload = tar_bytes([("main.tex", TEX), ("image.png", b"large image")])
                Path(destination).write_bytes(payload)
                return len(payload)
        collect_one(Client(), {"arxiv_id": "2603.20580"}, 1, output, state)
        self.assertFalse(seen[0].is_relative_to(output))
        self.assertEqual((output / "2603-20580/v1/main.tex").read_bytes(), TEX)


### 7c. Resume and reporting

Check interruption recovery, receipts, progress percentages, and reporting files.

In [ ]:
class ResumeAndReportingTests(DownloadTestBase):
    def test_interruption_preserves_previous_version_receipt(self):
        manifest = self.root / "manifest.json"
        manifest.write_text(json.dumps([{"id": "2603.20580", "versions": 2}]), encoding="utf-8")
        class InterruptSession(FakeSession):
            def get(self, url, **kwargs):
                if len(self.calls) == 1:
                    raise KeyboardInterrupt()
                return super().get(url, **kwargs)
        first = download_sources(manifest, self.root / "out", self.root / "state",
                             session=InterruptSession([FakeResponse(tar_bytes([("main.tex", TEX)]))]),
                             sleeper=lambda s: None)
        self.assertTrue(first["interrupted"])
        self.assertEqual(first["progress"]["versions_percent"], 50.0)
        later = download_sources(manifest, self.root / "out", self.root / "state",
                             session=FakeSession([FakeResponse(tar_bytes([("main.tex", TEX)]))]),
                             sleeper=lambda s: None)
        self.assertEqual(later["versions_skipped"], 1)
        self.assertEqual(later["progress"]["versions_percent"], 100.0)

    def test_resume_verifies_receipt_and_reports_percent(self):
        manifest = self.root / "manifest.json"
        manifest.write_text(json.dumps([{"arxiv_id": "2603.20580", "version_count": 2}]), encoding="utf-8")
        out, state = self.root / "output", self.root / "state"
        session = FakeSession([FakeResponse(tar_bytes([("main.tex", TEX)])), FakeResponse(b"%PDF-1.7\n")])
        first = download_sources(manifest, out, state, session=session, delay=3, sleeper=lambda s: None)
        self.assertEqual(first["progress"]["papers_percent"], 100.0)
        self.assertEqual(first["progress"]["versions_percent"], 100.0)
        self.assertEqual(first["versions_ok"], 1)
        self.assertEqual(first["versions_empty"], 1)
        with (state / "papers.csv").open(newline="", encoding="utf-8") as stream:
            row = next(csv.DictReader(stream))
        self.assertEqual(row["id"], "2603.20580")
        self.assertEqual(row["versions"], "2")
        self.assertEqual(row["versions_ok"], "1")
        self.assertEqual(row["size_after"], str(len(TEX)))
        self.assertEqual(row["status"], "partial")
        again = download_sources(manifest, out, state, session=FakeSession([]), delay=3, sleeper=lambda s: None)
        self.assertEqual(again["versions_skipped"], 2)
        (out / "2603-20580/v1/main.tex").write_bytes(b"corrupt")
        repaired = download_sources(manifest, out, state, session=FakeSession([FakeResponse(tar_bytes([("main.tex", TEX)]))]), delay=3, sleeper=lambda s: None)
        self.assertEqual(repaired["versions_skipped"], 1)
        self.assertEqual((out / "2603-20580/v1/main.tex").read_bytes(), TEX)

    def test_resume_check_skips_versions_without_receipt(self):
        state = self.root / "state"
        (state / "receipts" / "2603-20582").mkdir(parents=True)          # per-version receipt files
        (state / "receipts.jsonl").write_text(json.dumps({"arxiv_id": "2603.20580", "version": 1}) + "\n",
                                             encoding="utf-8")
        records = [{"arxiv_id": f"2603.{20580 + i}", "version_count": 2} for i in range(4)]
        checked = []
        def spy(state_dir, output_dir, arxiv_id, version, **kwargs):
            checked.append((arxiv_id, version))
        with patch.dict(globals(), verified_receipt=spy):
            self.assertEqual(load_receipts(records, self.root / "output", state), {})
        self.assertEqual(checked, [("2603.20580", 1), ("2603.20580", 2), ("2603.20582", 1), ("2603.20582", 2)])

    def test_failed_version_is_retried_on_next_run(self):
        manifest = self.root / "manifest.json"
        manifest.write_text(json.dumps([{"arxiv_id": "2603.20580", "version_count": 1}]), encoding="utf-8")
        output, state = self.root / "output", self.root / "state"
        failed = download_sources(manifest, output, state, session=FakeSession([FakeResponse(b"missing", 404)]), sleeper=lambda s: None)
        self.assertEqual(failed["progress"]["versions_percent"], 0.0)
        self.assertEqual(failed["versions_failed_this_run"], 1)
        recovered = download_sources(manifest, output, state, session=FakeSession([FakeResponse(tar_bytes([("main.tex", TEX)]))]), sleeper=lambda s: None)
        self.assertEqual(recovered["progress"]["versions_percent"], 100.0)
        self.assertGreaterEqual(recovered["disk_peak_estimate_bytes"], recovered["output_bytes"])
        history = (state / "run_history.jsonl").read_text(encoding="utf-8").splitlines()
        self.assertEqual(len(history), 2)
        self.assertEqual([json.loads(line)["progress"]["versions_percent"] for line in history], [0.0, 100.0])

### 7d. Filtering and pipelined download

Check that only `.tex`/`.bib` are kept, that older `.bbl`/`.sty`/`.cls` files are pruned on resume, that the next request is sent while the previous archive is still being processed, and that resume works from the `receipts.jsonl` index.

In [ ]:
import errno
import shutil
import threading
import time


class FilteringAndPerformanceTests(DownloadTestBase):
    def write_manifest(self, versions=1):
        manifest = self.root / "manifest.json"
        manifest.write_text(json.dumps([{"id": "2603.20580", "versions": versions}]), encoding="utf-8")
        return manifest

    def test_only_tex_and_bib_are_kept(self):
        raw = self.root / "raw"
        raw.write_bytes(tar_bytes([("main.tex", TEX), ("refs.bib", b"@book{a}"), ("main.bbl", b"bbl"),
                                   ("style.sty", b"sty"), ("class.cls", b"cls"), ("fig.png", b"image")]))
        info = extract_source(raw, self.root / "v1")
        self.assertEqual(sorted(info["files"]), ["main.tex", "refs.bib"])
        self.assertEqual(sorted(p.name for p in (self.root / "v1").rglob("*") if p.is_file()), ["main.tex", "refs.bib"])

    def test_bib_without_tex_is_empty_source(self):
        raw = self.root / "raw"
        raw.write_bytes(tar_bytes([("refs.bib", b"@book{a}"), ("paper.pdf", b"%PDF")]))
        info = extract_source(raw, self.root / "v1")
        self.assertEqual(info["status"], "empty_source")
        self.assertEqual(list((self.root / "v1").rglob("*")), [])

    def test_resume_prunes_legacy_bbl_without_new_request(self):
        manifest, out, state = self.write_manifest(), self.root / "out", self.root / "state"
        folder = out / "2603-20580/v1"
        (folder / "sub").mkdir(parents=True)
        (folder / "main.tex").write_bytes(TEX)
        (folder / "sub/main.bbl").write_bytes(b"legacy")
        legacy = {"arxiv_id": "2603.20580", "version": 1, "status": "ok", "before_bytes": 100,
                  "after_bytes": len(TEX) + 6, "downloaded_bytes": 50,
                  "files": {"main.tex": {"size": len(TEX), "sha256": hashlib.sha256(TEX).hexdigest()},
                            "sub/main.bbl": {"size": 6, "sha256": hashlib.sha256(b"legacy").hexdigest()}}}
        _atomic_json(_receipt_path(state, "2603.20580", 1), legacy)
        summary = download_sources(manifest, out, state, session=FakeSession([]), sleeper=lambda s: None)
        self.assertEqual(summary["versions_skipped"], 1)
        self.assertEqual(summary["source_after_bytes"], len(TEX))
        self.assertEqual([p.name for p in folder.rglob("*")], ["main.tex"])
        again = download_sources(manifest, out, state, session=FakeSession([]), sleeper=lambda s: None)
        self.assertEqual(again["versions_skipped"], 1)

    def test_next_fetch_overlaps_previous_processing(self):
        manifest, out, state = self.write_manifest(2), self.root / "out", self.root / "state"
        events, second_fetch = [], threading.Event()
        class Session(FakeSession):
            def get(self, url, **kwargs):
                events.append("fetch " + url[-2:])
                if url.endswith("v2"):
                    second_fetch.set()
                return super().get(url, **kwargs)
        original = process_download
        def slow_process(raw, identifier, version, *args):
            if version == 1:
                second_fetch.wait(5)
            receipt = original(raw, identifier, version, *args)
            events.append(f"processed v{version}")
            return receipt
        session = Session([FakeResponse(tar_bytes([("main.tex", TEX)])), FakeResponse(tar_bytes([("main.tex", TEX)]))])
        with patch.dict(globals(), process_download=slow_process):
            summary = download_sources(manifest, out, state, session=session, sleeper=lambda s: None)
        self.assertEqual(summary["progress"]["versions_percent"], 100.0)
        self.assertLess(events.index("fetch v2"), events.index("processed v1"))
        # both raw archives were on local disk at the same time
        self.assertEqual(summary["local_temp_peak_bytes"], 2 * len(tar_bytes([("main.tex", TEX)])))

    def test_progress_callback_and_bars(self):
        payload = tar_bytes([("main.tex", TEX)])
        seen = []
        client = SourceClient(FakeSession([FakeResponse(payload, headers={"Content-Length": str(len(payload))})]),
                              sleeper=lambda s: None)
        client.fetch("2603.20580", 1, self.root / "raw", progress=lambda done, total: seen.append((done, total)))
        self.assertEqual((seen[0], seen[-1]), ((0, len(payload)), (len(payload), len(payload))))
        manifest, out, state = self.write_manifest(2), self.root / "out", self.root / "state"
        summary = download_sources(manifest, out, state, progress=True, sleeper=lambda s: None,
                                   session=FakeSession([FakeResponse(payload), FakeResponse(payload)]))
        self.assertEqual(summary["progress"]["versions_percent"], 100.0)

    def test_resume_uses_receipt_index_and_sha256_mode_detects_same_size_damage(self):
        manifest, out, state = self.write_manifest(), self.root / "out", self.root / "state"
        download_sources(manifest, out, state, session=FakeSession([FakeResponse(tar_bytes([("main.tex", TEX)]))]),
                     sleeper=lambda s: None)
        shutil.rmtree(state / "receipts")  # Index alone is enough for a fast resume.
        with (state / "receipts.jsonl").open("a", encoding="utf-8") as stream:
            stream.write('{"torn": ')
        self.assertEqual(download_sources(manifest, out, state, session=FakeSession([]),
                                      sleeper=lambda s: None)["versions_skipped"], 1)
        (out / "2603-20580/v1/main.tex").write_bytes(b"x" * len(TEX))
        self.assertEqual(download_sources(manifest, out, state, session=FakeSession([]),
                                      sleeper=lambda s: None)["versions_skipped"], 1)
        repaired = download_sources(manifest, out, state, verify="sha256", sleeper=lambda s: None,
                                session=FakeSession([FakeResponse(tar_bytes([("main.tex", TEX)]))]))
        self.assertEqual(repaired["versions_skipped"], 0)
        self.assertEqual((out / "2603-20580/v1/main.tex").read_bytes(), TEX)

class RobustnessTests(DownloadTestBase):
    def manifest(self, versions):
        path = self.root / "manifest.json"
        path.write_text(json.dumps([{"id": f"2603.{20580 + i:05d}", "versions": 1} for i in range(versions)]),
                        encoding="utf-8")
        return path

    def test_stops_after_consecutive_failures(self):
        session = FakeSession([FakeResponse(b"missing", 404) for _ in range(10)])
        summary = download_sources(self.manifest(10), self.root / "out", self.root / "state", session=session,
                                   sleeper=lambda s: None, max_consecutive_failures=3)
        self.assertIn("3 versions failed in a row", summary["stopped_reason"])
        self.assertEqual(len(session.calls), 3)            # the other 7 versions were not requested

    def test_storage_full_stops_at_once(self):
        def full_disk(*args):
            raise OSError(errno.ENOSPC, "No space left on device")
        payload = tar_bytes([("main.tex", TEX)])
        session = FakeSession([FakeResponse(payload) for _ in range(10)])
        with patch.dict(globals(), process_download=full_disk):
            summary = download_sources(self.manifest(10), self.root / "out", self.root / "state", workers=1,
                                       session=session, sleeper=lambda s: None)
        self.assertIn("storage error", summary["stopped_reason"])
        self.assertLessEqual(len(session.calls), 3)        # only the archives already in flight

    def test_lock_blocks_a_second_runtime(self):
        state = self.root / "state"
        state.mkdir()
        lock = state / "download.lock"
        lock.write_text(json.dumps({"owner": "other-host:1", "heartbeat": time.time()}), encoding="utf-8")
        with self.assertRaises(RuntimeError):
            download_sources(self.manifest(1), self.root / "out", state, session=FakeSession([]), sleeper=lambda s: None)
        lock.write_text(json.dumps({"owner": "other-host:1", "heartbeat": time.time() - 3600}), encoding="utf-8")
        summary = download_sources(self.manifest(1), self.root / "out", state, sleeper=lambda s: None,
                                   session=FakeSession([FakeResponse(tar_bytes([("main.tex", TEX)]))]))
        self.assertEqual(summary["progress"]["versions_percent"], 100.0)
        self.assertFalse(lock.exists())                    # released at the end

    def test_leftovers_of_a_killed_run_are_removed(self):
        out, state = self.root / "out", self.root / "state"
        (out / ".staging" / "tmpabc" / "source").mkdir(parents=True)
        (state / "replaced" / "2603-20580_v1_x").mkdir(parents=True)
        download_sources(self.manifest(1), out, state, sleeper=lambda s: None,
                         session=FakeSession([FakeResponse(tar_bytes([("main.tex", TEX)]))]))
        self.assertEqual(list((out / ".staging").iterdir()), [])
        self.assertEqual(list((state / "replaced").iterdir()), [])

    def test_free_space_is_checked_before_any_request(self):
        session = FakeSession([FakeResponse(tar_bytes([("main.tex", TEX)]))])
        tiny = type("Usage", (), {"free": 1024})()
        with patch.object(shutil, "disk_usage", return_value=tiny), self.assertRaises(RuntimeError):
            download_sources(self.manifest(1), self.root / "out", self.root / "state", session=session,
                             sleeper=lambda s: None)
        self.assertEqual(session.calls, [])
        with patch.object(shutil, "disk_usage", return_value=tiny):
            summary = download_sources(self.manifest(1), self.root / "out", self.root / "state", session=session,
                                       sleeper=lambda s: None, check_space=False)
        self.assertEqual(summary["progress"]["versions_percent"], 100.0)

### 7e. Run the offline tests

Runs the 33 tests when `RUN_TEST` is `True`. A simulated 404 prints a `FAILED` line and the
interruption test prints `Interrupted`; the final `OK` is the pass criterion.

In [ ]:
if RUN_TEST:
    suite = unittest.TestSuite()
    for case in (ManifestAndPacingTests, ArchiveAndSafetyTests, ResumeAndReportingTests,
                 FilteringAndPerformanceTests, RobustnessTests):
        suite.addTests(unittest.defaultTestLoader.loadTestsFromTestCase(case))
    result = unittest.TextTestRunner(verbosity=2).run(suite)
    if not result.wasSuccessful():
        raise AssertionError("Downloader offline tests failed")

## 8. Output and resume

Sources: `MyDrive/arxiv-project/data/raw/<yymm-id>/vN/` (only `.tex`/`.bib`). Receipts (one file per
version plus the `receipts.jsonl` index), `errors.jsonl`, `progress.json`, `papers.csv`,
`run_summary.json` and `run_history.jsonl`: `state/download/`. After a Colab disconnect, run
`05_run_pipeline.ipynb` again: the crawl resumes from the receipts. Use one downloader at a time per
Drive folder.